# In this notebook, we analyze RQ1:  How do textual features (framing devices) relate to interpretive frames?

Descriptive:
- Rates of each gpt label (frames and binary variables) by themselves and w.r.t political leaning

- Segmented frames by "gpt-topic", which was a variable labeled in the dataset already. There is theoretical reason to believe that coverage differs by War/Social Issue/Labor, but we do not find evidence for it


In [11]:
import os

file_path = "full_data.csv"

# Get size in bytes
file_size = os.path.getsize(file_path)
print(f"File size: {file_size} bytes")

File size: 84261234 bytes


In [12]:
import pandas as pd

labels_df = pd.read_csv('labels_df.csv')
df = pd.read_csv('truncated_samples.csv.gz')

labels_df.head(1)


,uuid,title,protest_excerpt,gpt_title_frame,gpt_excerpt_primary_frame,gpt_excerpt_secondary_frame,gpt_demands_present,gpt_protester_quote,gpt_authority_quote,gpt_thematic_present,frames_error,binary_error
0,0057efe4-f7bc-4c05-a7e3-3dcbe2dc8733,Ethiopia declares state of emergency as region...,ETHIOPIA’S AMHARA ETHNIC GROUP ACCUSES BIDEN O...,residual,disruption,harm_to_protesters,no,no,yes,yes,NaN,NaN


In [18]:
counts = pd.concat([
    labels_df['gpt_title_frame'].value_counts().rename('Title'),
    labels_df['gpt_excerpt_primary_frame'].value_counts().rename('Excerpt Primary'),
    labels_df['gpt_excerpt_secondary_frame'].value_counts().rename('Excerpt Secondary')
], axis=1)

counts

,Title,Excerpt Primary,Excerpt Secondary
residual,5244,350,NaN
grievance,930,2441,1846.0
disruption,865,1658,1277.0
protester_violence,628,1907,458.0
harm_to_protesters,551,1145,430.0
impactful,443,782,579.0
widespread_support,397,443,768.0
property_damage,181,403,331.0
peaceful,44,154,313.0


In [14]:
labels_df['gpt_excerpt_secondary_frame'].isna().sum()

np.int64(3281)

# preprocessing: merging GPT labels with original df, transforming binary cols

In [5]:
# df = original dataset
# labels_df = GPT frame annotation results

merged_df = df.merge(
    labels_df.drop(columns=["title", "protest_excerpt"]),
    on="uuid",
    how="left",
    validate="one_to_one"
)


binary_cols = [
    "gpt_demands_present",
    "gpt_protester_quote",
    "gpt_authority_quote",
    "gpt_thematic_present"
]

# change binary to 1s and 0s
merged_df[binary_cols] = (
    merged_df[binary_cols]
    .apply(lambda col: col.str.strip().str.lower())
    .replace({"yes": 1, "no": 0})
)



print("Original rows:", len(df))
print("GPT label rows:", len(labels_df))
print("Merged rows:", len(merged_df))

merged_df.to_csv("full_data.csv", index=False)

Original rows: 9283
GPT label rows: 9283
Merged rows: 9283


In [2]:
import pandas as pd

merged_df = pd.read_csv('full_data.csv')
merged_df.head(1)

,row_id,text-topic,text-topic-exp,text-entity-name,text-entity-gender,text-entity-sentiment,text-entity-sentiment-exp,text-generic-frame,text-generic-frame-exp,text-issue-frame,...,excerpt_length,gpt_title_frame,gpt_excerpt_primary_frame,gpt_excerpt_secondary_frame,gpt_demands_present,gpt_protester_quote,gpt_authority_quote,gpt_thematic_present,frames_error,binary_error
0,1,Ethiopia's Political Instability and State of ...,The article discusses the declaration of a sta...,Ethiopia's Amhara Region,NaN,Negative,The article discusses a state of emergency bei...,"['Security and defense', 'Economic', 'Policy p...",The article discusses a state of emergency dec...,Political Instability and Security Concerns,...,1321,residual,disruption,harm_to_protesters,0,0,1,1,NaN,NaN


## What are the rates of the binary variables, segmented by primary frame? 

In [14]:
binary_cols = [
    "gpt_demands_present",
    "gpt_protester_quote",
    "gpt_authority_quote",
    "gpt_thematic_present"
]


frame_binary_rates = (
    merged_df
    .groupby("gpt_excerpt_primary_frame")[binary_cols]
    .mean()
    .mul(100)
    .round(1)
)

frame_binary_rates

,gpt_demands_present,gpt_protester_quote,gpt_authority_quote,gpt_thematic_present
gpt_excerpt_primary_frame,,,,
disruption,81.7,53.8,78.0,52.8
grievance,98.0,69.7,70.7,66.5
harm_to_protesters,79.4,47.5,78.1,73.2
impactful,74.7,42.2,80.9,74.2
peaceful,74.7,50.0,76.6,44.8
property_damage,76.7,37.5,86.8,65.5
protester_violence,71.5,43.3,87.8,50.8
residual,36.6,20.9,82.0,42.6
widespread_support,85.3,49.2,72.5,63.0


From a descriptive first look:
- residual is anomalously low for demands present (makes sense given grievance was a frame option) and protester quote...
- grievance has by far highest rate of protester quotes
- property damage and protester violence have highest rate of authority quote
- protester_violence is lowest in demands present and is significantly lower than harm to protesters, aside from residual

In [15]:
binary_by_lean = (
    merged_df
    .groupby("political_leaning")[binary_cols]
    .mean()
    .mul(100)
    .round(1)
    .reindex(lean_order)
    .T
)

binary_by_lean

political_leaning,left,left_lean,center,right_lean,right
gpt_demands_present,77.3,82.7,78.4,83.0,80.0
gpt_protester_quote,47.0,56.1,42.4,49.5,51.1
gpt_authority_quote,77.3,77.3,76.2,79.4,82.1
gpt_thematic_present,66.3,59.9,62.7,62.9,59.4


Overall, protester quotes are significantly less common than authority quotes. 

# What about political leaning?

In [21]:
merged_df['political_leaning'].value_counts()

political_leaning
left_lean     4178
right         2441
center        1152
right_lean     921
left           591
Name: count, dtype: int64

In [8]:
lean_order = [
    "left",
    "left_lean",
    "center",
    "right_lean",
    "right"
]

frame_by_lean = pd.crosstab(
    merged_df["gpt_excerpt_primary_frame"],
    merged_df["political_leaning"],
    normalize="columns"
).mul(100).round(1)
# change to normalize="index" for rows sum to 100%

frame_by_lean = frame_by_lean.reindex(columns=lean_order)

frame_by_lean

political_leaning,left,left_lean,center,right_lean,right
gpt_excerpt_primary_frame,,,,,
disruption,8.1,16.3,16.6,17.8,23.6
grievance,22.5,31.4,24.3,27.5,19.0
harm_to_protesters,16.9,12.8,10.4,10.9,11.9
impactful,8.6,8.0,12.2,11.1,6.3
peaceful,2.0,2.0,1.5,0.9,1.3
property_damage,1.7,4.0,3.2,6.3,5.4
protester_violence,28.6,17.4,18.7,17.9,25.9
residual,5.8,2.7,6.6,3.8,3.8
widespread_support,5.8,5.5,6.6,3.9,2.8


Note: columns sum to 100%. 

- We see that disruption rank orders, with more conservative outlets having higher rates.
- protester violence is quite frequent for both extremes of the political spectrum

In [26]:

binary_by_lean = (
    merged_df
    .groupby("political_leaning")[binary_cols]
    .mean()
    .mul(100)
    .round(1)
    .T
)

binary_by_lean = binary_by_lean.reindex(columns=lean_order)

binary_by_lean

political_leaning,left,left_lean,center,right_lean,right
gpt_demands_present,77.3,82.7,78.4,83.0,80.0
gpt_protester_quote,47.0,56.1,42.4,49.5,51.1
gpt_authority_quote,77.3,77.3,76.2,79.4,82.1
gpt_thematic_present,66.3,59.9,62.7,62.9,59.4


less clear trends here, but we do see right outlets have highest authority quote rates. 

# 1.3 How do War protests differ in coverage from Social Issues/Labor protests?
Boyle et al. finds that War protests receive more critical coverage in headlines by way of being more radical, but not in the article itself. They also do not find evidence that more radical protest types would be more episodic than thematic

In [10]:
with pd.option_context('display.max_rows', None):
    display(merged_df['gpt-topic'].value_counts())

gpt-topic
Politics                     4657
Legal                         682
Labor                         672
Social Issues                 483
Crime                         467
War                           382
Environment                   302
Business                      273
Sports                        203
Education                     187
Entertainment                 183
Immigration                   141
Culture                       119
Health                         60
Technology                     51
Media                          39
Economy                        38
Human Rights                   38
Social                         27
International                  21
no_topic                       19
Society                        16
International Relations        12
Housing                        11
Public Safety                  11
History                        10
Community                       7
Animal Rights                   7
Environmental                   7
Worl

In [89]:
topic_order = ["War", "Social Issues", "Labor", "Legal", "Politics"]

title_frame_by_topic = (
    pd.crosstab(
        merged_df["gpt_title_frame"],
        merged_df["gpt-topic"],
        normalize="columns"
    )
    .mul(100)
    .round(1)
    .reindex(columns=topic_order)
)

title_frame_by_topic

gpt-topic,War,Social Issues,Labor,Legal,Politics
gpt_title_frame,,,,,
disruption,8.4,6.6,13.5,0.6,9.5
grievance,12.8,9.3,18.3,5.1,7.8
harm_to_protesters,3.1,3.9,2.2,10.6,6.1
impactful,1.8,3.3,14.0,5.1,3.6
peaceful,1.3,1.0,0.1,0.1,0.4
property_damage,1.8,2.3,0.0,0.4,2.1
protester_violence,5.0,12.4,0.0,4.5,9.1
residual,60.7,58.4,38.8,72.1,57.4
widespread_support,5.0,2.7,12.9,1.3,4.1


Sample size issue with so many residuals. No clear story

In [90]:
excerpt_primary_by_topic = (
    pd.crosstab(
        merged_df["gpt_excerpt_primary_frame"],
        merged_df["gpt-topic"],
        normalize="columns"
    )
    .mul(100)
    .round(1)
    .reindex(columns=topic_order)
)

excerpt_primary_by_topic

gpt-topic,War,Social Issues,Labor,Legal,Politics
gpt_excerpt_primary_frame,,,,,
disruption,15.2,13.3,21.4,11.4,17.7
grievance,39.8,22.2,43.8,17.6,19.1
harm_to_protesters,7.3,8.9,3.3,18.8,14.3
impactful,3.4,6.6,19.9,10.0,6.5
peaceful,2.1,2.5,0.7,2.1,1.8
property_damage,4.7,5.4,0.0,1.3,4.7
protester_violence,16.5,35.8,0.4,25.5,27.0
residual,1.8,1.9,1.2,10.9,4.3
widespread_support,9.2,3.5,9.2,2.5,4.7


Contrary to what I'd expect, Social Issues more than doubles War in rate of protester violence framing

In [91]:
binary_cols = [
    "gpt_demands_present",
    "gpt_protester_quote",
    "gpt_authority_quote",
    "gpt_thematic_present"
]

binary_by_topic = (
    merged_df
    .groupby("gpt-topic")[binary_cols]
    .mean()
    .mul(100)
    .round(1)
    .T
    .reindex(columns=topic_order)
)

binary_by_topic

gpt-topic,War,Social Issues,Labor,Legal,Politics
gpt_demands_present,88.0,78.9,90.3,70.4,80.4
gpt_protester_quote,53.4,49.1,78.3,34.5,50.7
gpt_authority_quote,80.6,80.1,68.3,81.5,82.2
gpt_thematic_present,62.6,72.5,56.5,52.5,61.1


In [92]:
frames = [
    "protester_violence",
    "harm_to_protesters",
    "property_damage",
    "disruption",
    "grievance",
    "widespread_support",
    "peaceful",
    "impact",
    "residual"
]

combined_results = {}

for topic in topic_order:
    subset = merged_df[merged_df["gpt-topic"] == topic]

    combined_results[topic] = {
        frame: (
            (
                (subset["gpt_excerpt_primary_frame"] == frame) |
                (subset["gpt_excerpt_secondary_frame"] == frame)
            ).mean() * 100
        )
        for frame in frames
    }

excerpt_any_frame_by_topic = (
    pd.DataFrame(combined_results)
    .round(1)
    .reindex(columns=topic_order)
)

excerpt_any_frame_by_topic

,War,Social Issues,Labor,Legal,Politics
protester_violence,23.0,42.7,0.4,33.0,32.3
harm_to_protesters,12.6,12.4,4.6,23.8,19.6
property_damage,8.1,8.5,0.0,3.5,9.1
disruption,25.4,31.5,42.1,22.1,32.3
grievance,63.6,36.6,71.1,24.9,39.6
widespread_support,19.9,8.9,27.2,6.6,11.5
peaceful,6.8,6.6,1.5,5.4,5.4
impact,0.0,0.0,0.0,0.0,0.0
residual,1.8,1.9,1.2,10.9,4.3


This table combines primary and secondary: it does not distinguish between the two. The columns will add up to more than 100%. 